In [1]:
import sqlite3

conn = sqlite3.connect("练习.db")
cur = conn.cursor()

# cur.execute("""
# CREATE TABLE employees (
#     id       INTEGER PRIMARY KEY,
#     name     TEXT,
#     age      INTEGER,
#     dept_id  INTEGER
# )
# """)

cur.execute("INSERT INTO employees (id, name, age, dept_id) VALUES (1, '张三', 28, 1)")
cur.execute("INSERT INTO employees (id, name, age, dept_id) VALUES (2, '李四', 32, 2)")
cur.execute("INSERT INTO employees (id, name, age, dept_id) VALUES (3, '王五', 25, 1)")
#cur.execute("INSERT INTO employees (id, name, age, dept_id) VALUES (1, '老六', 28, 1)")#主键重复（id=1），数据库数据重复，会报错
conn.commit()

cur.execute("SELECT * FROM employees")
for row in cur.fetchall():
    print(row)


IntegrityError: UNIQUE constraint failed: employees.id

In [1]:
import sqlite3

conn = sqlite3.connect("练习.db")
cur = conn.cursor()

# 1. 删掉旧表（连数据一起删）
cur.execute("DROP TABLE IF EXISTS employees")

# 2. 重新建表
cur.execute("""
CREATE TABLE employees (
    id       INTEGER PRIMARY KEY,
    name     TEXT,
    age      INTEGER,
    dept_id  INTEGER
)
""")

# 3. 重新插入
cur.execute("INSERT INTO employees (id, name, age, dept_id) VALUES (1, '张三', 28, 1)")
cur.execute("INSERT INTO employees (id, name, age, dept_id) VALUES (2, '李四', 32, 2)")
cur.execute("INSERT INTO employees (id, name, age, dept_id) VALUES (3, '王五', 25, 1)")

# 4. 保存并关闭
conn.commit()
conn.close()
print("数据已重建完成")


数据已重建完成


In [1]:
import sqlite3
conn = sqlite3.connect("练习.db")
cur = conn.cursor()

# 1. 查所有
cur.execute("SELECT * FROM employees")
print("=== 查所有 ===")
for row in cur.fetchall():
    print(row)

# 2. 只查名字和年龄
cur.execute("SELECT name, age FROM employees")
print("=== 只查名字年龄 ===")
for row in cur.fetchall():
    print(row)

# 3. 筛选年龄>28
cur.execute("SELECT name, age FROM employees WHERE age > 28")
print("=== 年龄>28 ===")
for row in cur.fetchall():
    print(row)

# 4. 按年龄排序（默认从小到大）
cur.execute("SELECT name, age FROM employees ORDER BY age")
print("=== 年龄从小到大 ===")
for row in cur.fetchall():
    print(row)

# 5. 找年龄最大的人
cur.execute("SELECT name, age FROM employees ORDER BY age DESC LIMIT 1")
print("=== 年龄最大 ===")
for row in cur.fetchall():
    print(row)

conn.close()


=== 查所有 ===
(1, '张三', 28, 1)
(2, '李四', 32, 2)
(3, '王五', 25, 1)
=== 只查名字年龄 ===
('张三', 28)
('李四', 32)
('王五', 25)
=== 年龄>28 ===
('李四', 32)
=== 年龄从小到大 ===
('王五', 25)
('张三', 28)
('李四', 32)
=== 年龄最大 ===
('李四', 32)


In [2]:
import sqlite3
conn = sqlite3.connect("练习.db")
cur = conn.cursor()

cur.execute("INSERT INTO employees (id, name, age, dept_id) VALUES (4, '赵六', 35, 1)")
cur.execute("INSERT INTO employees (id, name, age, dept_id) VALUES (5, '钱七', 29, 2)")
cur.execute("INSERT INTO employees (id, name, age, dept_id) VALUES (6, '孙八', 41, 3)")
cur.execute("INSERT INTO employees (id, name, age, dept_id) VALUES (7, '周九', 26, 3)")

conn.commit()
conn.close()
print("已添加4人")


已添加4人


In [3]:
import sqlite3
conn = sqlite3.connect("练习.db")
cur = conn.cursor()

# 1. 一共多少人
cur.execute("SELECT COUNT(*) FROM employees")
print("总人数:", cur.fetchone()[0])

# 2. 平均年龄
cur.execute("SELECT AVG(age) FROM employees")
print("平均年龄:", cur.fetchone()[0])

# 3. 最大/最小年龄
cur.execute("SELECT MAX(age), MIN(age) FROM employees")
print("最大最小:", cur.fetchone())

# 4. 按部门分组，统计每个部门多少人
cur.execute("SELECT dept_id, COUNT(*) FROM employees GROUP BY dept_id")
print("=== 每个部门人数 ===")
for row in cur.fetchall():
    print(f"部门{row[0]}: {row[1]}人")

# 5. 每个部门平均年龄
cur.execute("SELECT dept_id, AVG(age) FROM employees GROUP BY dept_id")
print("=== 每个部门平均年龄 ===")
for row in cur.fetchall():
    print(f"部门{row[0]}: 平均{row[1]:.1f}岁")

# 6. 分组后再过滤：人数>=2的部门
cur.execute("SELECT dept_id, COUNT(*) FROM employees GROUP BY dept_id HAVING COUNT(*) >= 2")
print("=== 人数>=2的部门 ===")
for row in cur.fetchall():
    print(f"部门{row[0]}: {row[1]}人")

conn.close()


总人数: 7
平均年龄: 30.857142857142858
最大最小: (41, 25)
=== 每个部门人数 ===
部门1: 3人
部门2: 2人
部门3: 2人
=== 每个部门平均年龄 ===
部门1: 平均29.3岁
部门2: 平均30.5岁
部门3: 平均33.5岁
=== 人数>=2的部门 ===
部门1: 3人
部门2: 2人
部门3: 2人


In [4]:
import sqlite3
conn = sqlite3.connect("练习.db")
cur = conn.cursor()

# A. WHERE：先过滤行，再分组
#    先把部门3的人筛掉（只剩部门1、2的人），再按部门统计人数
cur.execute("SELECT dept_id, COUNT(*) FROM employees WHERE dept_id <> 3 GROUP BY dept_id")
print("=== WHERE过滤后（排除部门3）分组 ===")
for row in cur.fetchall():
    print(f"部门{row[0]}: {row[1]}人")

# B. HAVING：先全部分组统计，再过滤组
#    所有部门都统计，然后只保留人数>=2的部门
cur.execute("SELECT dept_id, COUNT(*) FROM employees GROUP BY dept_id HAVING COUNT(*) >= 2")
print("=== HAVING过滤组（人数>=2的部门）===")
for row in cur.fetchall():
    print(f"部门{row[0]}: {row[1]}人")

conn.close()


=== WHERE过滤后（排除部门3）分组 ===
部门1: 3人
部门2: 2人
=== HAVING过滤组（人数>=2的部门）===
部门1: 3人
部门2: 2人
部门3: 2人


In [5]:
import sqlite3
conn = sqlite3.connect("练习.db")
cur = conn.cursor()

# A. WHERE按age筛行：各部门30岁以上的员工数
cur.execute("SELECT dept_id, COUNT(*) FROM employees WHERE age > 30 GROUP BY dept_id")
print("=== 各部门 年龄>30 的人数（WHERE先筛人）===")
for row in cur.fetchall():
    print(f"部门{row[0]}: {row[1]}人")

# B. HAVING按age筛组：平均年龄>=30的部门
cur.execute("SELECT dept_id, AVG(age) FROM employees GROUP BY dept_id HAVING AVG(age) >= 30")
print("=== 平均年龄>=30的部门（HAVING筛组）===")
for row in cur.fetchall():
    print(f"部门{row[0]}: 平均{row[1]:.1f}岁")

conn.close()


=== 各部门 年龄>30 的人数（WHERE先筛人）===
部门1: 1人
部门2: 1人
部门3: 1人
=== 平均年龄>=30的部门（HAVING筛组）===
部门2: 平均30.5岁
部门3: 平均33.5岁


In [6]:
import sqlite3
conn = sqlite3.connect("练习.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE departments (
    id    INTEGER PRIMARY KEY,
    name  TEXT
)
""")

cur.execute("INSERT INTO departments (id, name) VALUES (1, '研发部')")
cur.execute("INSERT INTO departments (id, name) VALUES (2, '生产部')")
cur.execute("INSERT INTO departments (id, name) VALUES (3, '质检部')")

conn.commit()
conn.close()
print("部门表建好了")


部门表建好了


In [7]:
import sqlite3
conn = sqlite3.connect("练习.db")
cur = conn.cursor()

cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
for row in cur.fetchall():
    print(row[0])

conn.close()


employees
departments


In [8]:
import sqlite3
conn = sqlite3.connect("练习.db")
cur = conn.cursor()

cur.execute("""
SELECT employees.name, departments.name
FROM employees
INNER JOIN departments ON employees.dept_id = departments.id
""")
print("=== 员工及其部门 ===")
for row in cur.fetchall():
    print(row)

conn.close()


=== 员工及其部门 ===
('张三', '研发部')
('李四', '生产部')
('王五', '研发部')
('赵六', '研发部')
('钱七', '生产部')
('孙八', '质检部')
('周九', '质检部')


In [9]:
import sqlite3
conn = sqlite3.connect("练习.db")
cur = conn.cursor()

cur.execute("""
SELECT employees.name, employees.age, departments.name
FROM employees
INNER JOIN departments ON employees.dept_id = departments.id
""")
print("=== 员工：名字+年龄+部门 ===")
for row in cur.fetchall():
    print(row)

conn.close()


=== 员工：名字+年龄+部门 ===
('张三', 28, '研发部')
('李四', 32, '生产部')
('王五', 25, '研发部')
('赵六', 35, '研发部')
('钱七', 29, '生产部')
('孙八', 41, '质检部')
('周九', 26, '质检部')
